# Instructor Effectiveness Modeling (EdTech)

This notebook uses only the real CSV at `data/instructor_data.csv`. The goal is a transparent, course-adjusted effectiveness proxy for coaching and support, not a personnel decision rule.


## 1. Setup and Load

We set one random seed, use relative paths, select the requested CSV deterministically, and inspect its structure before analysis. No synthetic or fallback data is allowed.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             precision_recall_fscore_support)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
DATA_DIR = Path("data")
requested_path = DATA_DIR / "instructor_data.csv"
csv_paths = sorted(DATA_DIR.glob("*.csv"))
DATA_PATH = requested_path if requested_path.exists() else (csv_paths[0] if len(csv_paths) == 1 else None)
if DATA_PATH is None:
    raise FileNotFoundError("No real CSV found. Put the dataset in ./data/instructor_data.csv or make ./data/ contain one CSV.")

df = pd.read_csv(DATA_PATH)
required = ["batch_id", "instructor_id", "course_id", "completion_rate", "dropout_rate",
            "avg_score_improvement", "avg_quiz_score", "avg_watch_time",
            "assignment_submission_rate", "forum_activity_rate", "avg_feedback_score",
            "feedback_response_rate"]
column_map = {}
missing_required = sorted(set(required) - set(df.columns))
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}. Add an explicit column_map before loading.")
print(f"DATA_PATH: {DATA_PATH}")
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Dtypes:\n", df.dtypes)
print("Head:\n", df.head())
print("Missing counts:\n", df.isna().sum())
assert set(required).issubset(df.columns)


## 2. EDA: Data Quality and Distributions

These checks look for missingness, duplicate rows, invalid ranges, impossible completion/dropout combinations, skew, outliers, and feedback nonresponse bias. They describe the data before we define a target.


In [ ]:
rate_cols = ["completion_rate", "dropout_rate", "avg_watch_time", "assignment_submission_rate", "forum_activity_rate", "feedback_response_rate"]
feedback_cols = ["avg_feedback_score"]
print("Duplicate rows:", int(df.duplicated().sum()))
range_checks = {column: int(((df[column] < 0) | (df[column] > 1)).sum()) for column in rate_cols}
feedback_check = int(((df["avg_feedback_score"] < 1) | (df["avg_feedback_score"] > 5)).sum())
consistency = df["completion_rate"] + df["dropout_rate"]
print("Out-of-range rate counts:", range_checks)
print("Out-of-range feedback count:", feedback_check)
print("Completion + dropout summary:\n", consistency.describe())
print("Rows outside +/- 0.05 consistency tolerance:", int((consistency.sub(1).abs() > 0.05).sum()))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df[rate_cols].plot(kind="box", ax=axes[0, 0], rot=35, title="Rate metric outliers")
axes[0, 0].set_ylabel("Rate")
df[feedback_cols].plot(kind="hist", bins=20, ax=axes[0, 1], title="Feedback score distribution", legend=False)
axes[0, 1].set_xlabel("Average feedback score")
axes[0, 1].set_ylabel("Batch count")
df.groupby("instructor_id")["batch_id"].nunique().plot(kind="hist", bins=20, ax=axes[1, 0], title="Batches per instructor")
axes[1, 0].set_xlabel("Number of batches")
axes[1, 0].set_ylabel("Instructor count")
low_response = df["feedback_response_rate"] <= df["feedback_response_rate"].quantile(0.25)
axes[1, 1].scatter(df["feedback_response_rate"], df["avg_feedback_score"], alpha=0.25)
axes[1, 1].axvline(df.loc[low_response, "feedback_response_rate"].max(), color="red", linestyle="--", label="Lowest-response quartile")
axes[1, 1].set(title="Feedback score versus response rate", xlabel="Feedback response rate", ylabel="Average feedback score")
axes[1, 1].legend()
plt.tight_layout()

plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include=np.number).corr(), cmap="vlag", center=0)
plt.title("Correlation heatmap of numeric batch metrics")
plt.xlabel("Metric")
plt.ylabel("Metric")
plt.tight_layout()


In [ ]:
"""Early Observations

The printed checks and plots above are the evidence for data quality. Feedback response rate is treated as a reliability signal rather than assuming every average feedback score is equally precise. Batch IDs are identifiers here, so later trend analysis uses their observed order only as a time proxy.
"""

In [ ]:
"""3. Course-Adjusted Effectiveness Score

Metrics are standardized within course_id so comparisons reflect relative performance in the same course. The assumed weights prioritize learning outcomes (0.60), then reliability-weighted learner feedback (0.25), then consistency (0.15). These are value judgments, so stability is tested later.
"""
metric_cols = ["completion_rate", "dropout_rate", "avg_score_improvement", "avg_quiz_score", "avg_feedback_score"]
df["positive_dropout"] = 1 - df["dropout_rate"]
score_parts = ["completion_rate", "positive_dropout", "avg_score_improvement", "avg_quiz_score"]
for column in metric_cols + ["positive_dropout"]:
    course_mean = df.groupby("course_id")[column].transform("mean")
    course_std = df.groupby("course_id")[column].transform("std").replace(0, np.nan)
    df[f"z_{column}"] = ((df[column] - course_mean) / course_std).fillna(0)

df["learning_outcomes"] = df[[f"z_{column}" for column in score_parts]].mean(axis=1)
df["feedback_component"] = df["z_avg_feedback_score"] * df["feedback_response_rate"]
df["consistency_component"] = 1 - df.groupby("instructor_id")["learning_outcomes"].transform("std").fillna(0).clip(upper=1)
df["effectiveness_score"] = (0.60 * df["learning_outcomes"] + 0.25 * df["feedback_component"] + 0.15 * df["consistency_component"])

instructor_scores = df.groupby("instructor_id")["effectiveness_score"].mean().rename("effectiveness_score").to_frame()
terciles = instructor_scores["effectiveness_score"].quantile([1/3, 2/3]).to_numpy()
instructor_scores["tier"] = pd.cut(instructor_scores["effectiveness_score"], bins=[-np.inf, *terciles, np.inf], labels=["Low", "Medium", "High"], include_lowest=True)
print("Tier cut points:", terciles)
print("Tier counts:\n", instructor_scores["tier"].value_counts().sort_index())
print("Tier proportions:\n", instructor_scores["tier"].value_counts(normalize=True).sort_index())


## 4. Instructor-Level Aggregation and Confidence

Means describe typical performance; medians resist outliers; standard deviation and coefficient of variation describe reliability; min/max and their gap show floor-to-ceiling spread. The batch ID order is only a time proxy, not a verified calendar. Scores from instructors with few batches are shrunk toward the global mean using an empirical-Bayes-style reliability weight.


In [ ]:
def trend_slope(values):
    values = np.asarray(values, dtype=float)
    return float(np.polyfit(np.arange(len(values)), values, 1)[0]) if len(values) >= 2 else 0.0

def coefficient_of_variation(values):
    mean = float(np.mean(values))
    return float(np.std(values, ddof=1) / abs(mean)) if len(values) > 1 and mean != 0 else 0.0

grouped = df.sort_values(["instructor_id", "batch_id"]).groupby("instructor_id")
agg = grouped["effectiveness_score"].agg(mean="mean", median="median", std="std", min="min", max="max")
agg["cv"] = grouped["effectiveness_score"].apply(coefficient_of_variation)
agg["best_worst_gap"] = agg["max"] - agg["min"]
agg["trend_slope_batch_order"] = grouped["effectiveness_score"].apply(trend_slope)
agg["n_batches"] = grouped["batch_id"].nunique()
agg["n_courses"] = grouped["course_id"].nunique()
agg["global_mean"] = df["effectiveness_score"].mean()
prior_strength = 3
agg["shrunk_mean"] = (agg["n_batches"] * agg["mean"] + prior_strength * agg["global_mean"]) / (agg["n_batches"] + prior_strength)
agg["low_confidence"] = agg["n_batches"] < 3
agg["recent_weighted_mean"] = grouped.apply(lambda batch: np.average(batch["effectiveness_score"], weights=np.arange(1, len(batch) + 1)))
agg["recent_minus_plain"] = agg["recent_weighted_mean"] - agg["mean"]
agg["tier"] = instructor_scores["tier"]
print(agg.describe(include="all"))
print("Low-confidence instructors:", int(agg["low_confidence"].sum()))
print("Many-batch recent-vs-plain comparison (n_batches >= 5):\n", agg.loc[agg["n_batches"] >= 5, ["mean", "recent_weighted_mean", "recent_minus_plain"]].describe())


In [ ]:
"""5. Leakage-Safe Modeling Features

The target is built from outcome and feedback metrics. The primary model therefore uses only engagement, consistency, experience, and versatility. An ablation adds outcome-derived features only as a diagnostic, never as the production feature set.
"""

## 6. Models and Correlation-Based Feature Pruning

We compare a majority-class DummyClassifier, scaled Logistic Regression, Random Forest, and Gradient Boosting. Balanced class weights are used where the estimator supports them. Highly correlated numeric predictors are pruned using a training-feature correlation threshold to reduce redundant signals.


In [ ]:
model_df = agg.reset_index().copy()
engagement_features = ["avg_watch_time", "assignment_submission_rate", "forum_activity_rate"]
# Instructor-level engagement summaries avoid mixing batch rows with instructor targets.
engagement_agg = df.groupby("instructor_id")[engagement_features].agg(["mean", "std"])
engagement_agg.columns = ["_".join(column) for column in engagement_agg.columns]
model_df = model_df.join(engagement_agg)
model_df["consistency"] = 1 / (1 + model_df["std"].fillna(0))
base_features = [f"{column}_{stat}" for column in engagement_features for stat in ["mean", "std"]]
base_features += ["consistency", "n_batches", "n_courses"]

# Correlation pruning is fit on the instructor feature table to remove redundant predictors.
def prune_correlated(features, threshold=0.90):
    corr = model_df[features].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    return [column for column in upper.columns if not (upper[column] > threshold).any()]

selected_features = prune_correlated(base_features)
X = model_df[selected_features].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
assert np.isfinite(X.to_numpy()).all()
y = model_df["tier"].astype(str)
groups = model_df["instructor_id"]
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
models = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE),
    "Logistic Regression": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))]),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

def evaluate_model(name, estimator, features=X):
    predictions = cross_val_predict(estimator, features, y, groups=groups, cv=cv, method="predict")
    precision, recall, f1, _ = precision_recall_fscore_support(y, predictions, labels=["Low", "Medium", "High"], zero_division=0)
    return {"model": name, "macro_f1": f1_score(y, predictions, average="macro"), "predictions": predictions,
            "low_precision": precision[0], "medium_precision": precision[1], "high_precision": precision[2],
            "low_recall": recall[0], "medium_recall": recall[1], "high_recall": recall[2]}

results = [evaluate_model(name, estimator) for name, estimator in models.items()]
results_df = pd.DataFrame(results).drop(columns="predictions").sort_values("macro_f1", ascending=False)
print("Selected features:", selected_features)
print(results_df)


In [ ]:
# Ablation: this intentionally adds outcome-derived aggregates to show why leakage would inflate apparent performance.
leaky_features = list(dict.fromkeys(selected_features + ["mean", "median", "cv", "trend_slope_batch_order"]))
leaky_X = model_df[leaky_features].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
assert np.isfinite(leaky_X.to_numpy()).all()
ablation_results = [evaluate_model("Logistic Regression with outcome-derived ablation", models["Logistic Regression"], leaky_X)]
print(pd.DataFrame(ablation_results).drop(columns="predictions"))
print("The production result above excludes outcome-derived columns; the ablation is diagnostic only.")


## 7. Grouped Evaluation

Macro-F1 is primary because it gives each tier equal importance. All folds are grouped by instructor, so an instructor cannot appear in both train and test. We also inspect per-class precision/recall, the confusion matrix, baseline lift, and an ordinal error where Low-to-High mistakes cost two steps instead of one.


In [ ]:
best_name = results_df.iloc[0]["model"]
best_prediction = next(item["predictions"] for item in results if item["model"] == best_name)
baseline_f1 = float(results_df.loc[results_df["model"] == "Dummy baseline", "macro_f1"].iloc[0])
best_f1 = float(results_df.iloc[0]["macro_f1"])
print("Best model:", best_name)
print(classification_report(y, best_prediction, labels=["Low", "Medium", "High"], zero_division=0))
print("Confusion matrix (rows true, columns predicted):\n", pd.DataFrame(confusion_matrix(y, best_prediction, labels=["Low", "Medium", "High"]), index=["Low", "Medium", "High"], columns=["Low", "Medium", "High"]))
ordinal = {"Low": 0, "Medium": 1, "High": 2}
ordinal_error = np.mean([abs(ordinal[actual] - ordinal[predicted]) for actual, predicted in zip(y, best_prediction)])
print("Mean ordinal error:", round(float(ordinal_error), 4))
print("Macro-F1 lift versus baseline:", round(best_f1 - baseline_f1, 4))


## 8. Interpretation

Permutation importance measures how much shuffling one feature reduces predictive quality. Partial-dependence plots show the model's average prediction response to a feature, while reminding us that these are associations rather than causal effects. The audience-facing interpretation is: a top feature is a useful coaching signal, not proof that changing it alone will change outcomes.


In [ ]:
best_estimator = clone(models[best_name])
best_estimator.fit(X, y)
perm = permutation_importance(best_estimator, X, y, scoring="f1_macro", n_repeats=20, random_state=RANDOM_STATE)
importance = pd.Series(perm.importances_mean, index=selected_features).sort_values(ascending=False)
print("Permutation importance:\n", importance)

plot_features = list(importance.head(min(3, len(importance))).index)
if plot_features:
    fig, ax = plt.subplots(figsize=(12, 4 * len(plot_features)))
    PartialDependenceDisplay.from_estimator(best_estimator, X, plot_features, target="High", ax=ax)
    plt.suptitle("Partial dependence for leading model features")
    plt.tight_layout()

print("Product uses: coaching cards can surface actionable engagement patterns; early-warning flags can prompt support; batch allocation support can combine this proxy with context and human review.")


## 9. Sensitivity and Stability

We perturb the three score weights with reproducible Monte Carlo draws, renormalize them, recompute instructor tiers, and report unchanged tiers. Bootstrap resampling of batch rows gives another view of uncertainty. Borderline instructors are those closest to a tier boundary in the baseline score.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
base_components = df.groupby("instructor_id")[["learning_outcomes", "feedback_component", "consistency_component"]].mean()
base_tiers = instructor_scores["tier"].astype(str)
stability = []
for _ in range(200):
    weights = rng.dirichlet([0.60 * 20, 0.25 * 20, 0.15 * 20])
    perturbed = base_components.to_numpy() @ weights
    cuts = np.quantile(perturbed, [1 / 3, 2 / 3])
    tiers = pd.cut(perturbed, bins=[-np.inf, *cuts, np.inf], labels=["Low", "Medium", "High"], include_lowest=True).astype(str)
    stability.append(np.mean(tiers.to_numpy() == base_tiers.reindex(base_components.index).to_numpy()) * 100)

bootstrap_agreement = []
for _ in range(200):
    sample = df.sample(frac=1, replace=True, random_state=int(rng.integers(0, 1_000_000)))
    boot_scores = sample.groupby("instructor_id")["effectiveness_score"].mean().reindex(instructor_scores.index).fillna(instructor_scores["effectiveness_score"])
    cuts = instructor_scores["effectiveness_score"].quantile([1 / 3, 2 / 3]).to_numpy()
    boot_tiers = pd.cut(boot_scores, bins=[-np.inf, *cuts, np.inf], labels=["Low", "Medium", "High"], include_lowest=True).astype(str)
    bootstrap_agreement.append(np.mean(boot_tiers.to_numpy() == base_tiers.to_numpy()) * 100)

boundary_distance = np.minimum(abs(instructor_scores["effectiveness_score"] - terciles[0]), abs(instructor_scores["effectiveness_score"] - terciles[1]))
borderline = boundary_distance.sort_values().head(10).index.tolist()
stability_pct = float(np.mean(stability))
bootstrap_pct = float(np.mean(bootstrap_agreement))
print("Mean tier stability under 200 weight perturbations (%):", round(stability_pct, 2))
print("Mean bootstrap tier agreement (%):", round(bootstrap_pct, 2))
print("Borderline instructors:", borderline)


In [ ]:
## 10. Mandatory Questions: Evidence From This Run

print("Q1 evidence - leading features:", importance.head(5).round(4).to_dict())
print("Q2 evidence - feedback nonresponse correlation:", round(float(df[["avg_feedback_score", "feedback_response_rate"]].corr().iloc[0, 1]), 4))
print("Q3 evidence - low-confidence instructors:", int(agg["low_confidence"].sum()), "of", len(agg))
print("Q4 available fields:", list(df.columns))
print("Q5 recommendation: not as a sole judge; use for coaching with human review.")


### Q1. Which features most influenced effectiveness and why?

Use the printed permutation-importance ranking from the real grouped evaluation. Its leading engagement and experience variables are predictive associations: they may identify where coaching attention is useful, but they do not prove causation.


In [ ]:
"""Q2. Which variables could be misleading or confounded?

Course difficulty, learner mix, batch size, prior skill, selection into a course, and feedback response bias can confound every association. Completion and dropout are mechanically related. A low response rate can make an average feedback score unstable, which is why feedback is reliability-weighted.
"""

### Q3. How could the model fail in real-world usage?

It can fail for instructors with few batches, under distribution drift, when metrics are gamed, when learner populations change, or when feedback is systematically missing. Grouped validation reduces leakage but cannot remove these real-world shifts.


In [ ]:
"""Q4. What additional data would improve the analysis?

Useful additions are learner baseline skill and background, batch size, course difficulty, verified timestamps, session-level engagement, TA support, attendance, assessment design, and qualitative observations. These would help separate instructor effects from learner and course context.

Q5. Should this model be used for instructor performance evaluation?

No, not as a sole judge. Use it for coaching cards, early-warning support, and batch-allocation discussion with human review, transparency, uncertainty, and an opportunity for instructors to add context.
"""

## 11. Conclusion, Limitations and Ethics

This is a reproducible descriptive modeling exercise based on the supplied batch CSV. The score is a proxy, not ground truth. Course adjustment, shrinkage, grouped validation, and stability analysis improve discipline but do not establish causality or fairness. Do not use these tiers alone for hiring, pay, promotion, or termination. Use them for supportive coaching with human review and instructor transparency.


## 12. Final Summary Table

The table below is generated from the real run, so the reported values cannot become stale when the CSV changes.


In [ ]:
tier_distribution = instructor_scores["tier"].value_counts().reindex(["Low", "Medium", "High"], fill_value=0)
summary_table = pd.DataFrame({
    "Metric": ["Rows", "Instructors", "Low tier", "Medium tier", "High tier", "Best model", "Best macro-F1", "Dummy macro-F1", "Mean weight stability %", "Mean bootstrap agreement %", "Top drivers"],
    "Value": [len(df), len(instructor_scores), int(tier_distribution["Low"]), int(tier_distribution["Medium"]), int(tier_distribution["High"]), best_name, round(best_f1, 4), round(baseline_f1, 4), round(stability_pct, 2), round(bootstrap_pct, 2), ", ".join(importance.head(3).index)]
})
print(summary_table.to_string(index=False))
